In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, json, os, urllib.request, tempfile, time
root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
run = root/'phase2_results/final_sprint_v1'
source = run/'code'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(source/'requirements.txt')],check=True)
sys.path.insert(0,str(source))
from common import read, save, sha, metrics, context
from runtime import Pipeline, DEMO_IDS
from evaluate import benchmark
import numpy as np
import torch
assert sha(run/'policy.json')=='1ee0887f4e6ec960c421aff5d60af40d6c500698570bdeaa40b3fa06a1ae5328'
out = run.parent/'validation_followup'/str(time.time_ns())
out.mkdir(parents=True)
cached = Path(tempfile.mkdtemp(prefix='camera_demo_'))
(cached/'images').mkdir()
for ident in DEMO_IDS:
    (cached/'images'/(ident+'.jpg')).symlink_to(root/'images'/(ident+'.jpg'))
data=np.load(run/'classifier/validation_logits.npz')
scores=data['logits'][:,:14]
labels=data['labels']
counts=np.array(read(run/'data_support.json')['training_counts'][:14])
prior=np.log(counts/counts.sum())
species=[]
for strength in (0,.10,.25,.50):
    m=metrics(scores-strength*prior,labels,read(run/'policy.json')['species_temperature'])
    species.append(dict(strength=strength,macro_f1=m['macro_f1'],accuracy=m['accuracy'],ece=m['ece']))
save(out/'species_validation.json',species)
print('Species validation',species,flush=True)
_,_,rows,_=context(root)
candidate=next(r for r in read(run/'mask_selection.json')['finalists'] if r['name']=='m_v4_sam_t640')
policy=read(Path(candidate['policy']))
sam=policy['mask']['sam']
if not Path(sam['path']).exists():
    target=cached/'sam2.1_t.pt'
    urllib.request.urlretrieve('https://github.com/ultralytics/assets/releases/download/v8.3.0/sam2.1_t.pt',target)
    assert sha(target)==sam['sha256']
    sam['path']=str(target)
save(out/'candidate.json',policy)
timings=[]
for threads in sorted(set(min(n,len(os.sched_getaffinity(0))) for n in (1,2,4))):
    p=Pipeline(root,cached,out/'candidate.json','cpu')
    p.process_frame(cached/'images'/(DEMO_IDS[0]+'.jpg'),next(r for r in rows if r['id']==DEMO_IDS[0]))
    torch.set_num_threads(threads)
    measurement=benchmark(p,rows,cached,out/f'cpu_{threads}')
    timings.append(dict(requested_threads=threads,actual_threads=measurement['threads'],p95=measurement['p95']))
    del p
save(out/'summary.json',dict(species_validation=species,cpu=timings,previous_candidate_validation_iou=candidate['mean_iou'],scope='Validation follow-up only'))
print('Saved',out/'summary.json',timings)
